# Day 20/42: Cross Validation and Hyperparameter Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week3_core_ml/day20_cross_validation_tuning/day20_notebook.ipynb)

## What You'll Learn

- Why a single train/test split can hand you a lucky or unlucky number
- How K-Fold cross-validation gives you one trustworthy score instead of a guess
- How to tune a model properly with GridSearchCV and RandomizedSearchCV
- The data leakage trap that quietly inflates your CV score, and how to avoid it

## Dataset used

A synthetic classification dataset built in this notebook with `make_classification`, no download needed. Built deliberately noisy in places, because that's where these techniques actually earn their keep.

Run every cell top to bottom. Nothing here needs an internet connection or an API key.

## Setup

In [ ]:
%matplotlib inline
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform
from sklearn.datasets import make_classification
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                       GridSearchCV, RandomizedSearchCV)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

np.random.seed(42)
print("Setup complete. Libraries loaded.")

## The Concept

A single train/test split tells you how a model did on one particular slice of your data. Shuffle the split differently and the same model, same data, same code, can report a meaningfully different score. That's not your model changing. That's noise.

**K-Fold Cross-Validation**: split the data into K equal folds. Train on K-1 of them, validate on the one left out. Repeat K times so every row gets a turn as validation data. Average the K scores. You get one number with a sense of how much it varies, instead of one number that might just be luck.

**GridSearchCV**: tries every combination of hyperparameters you specify, scoring each with cross-validation, and returns the best one.

**RandomizedSearchCV**: samples a fixed number of random combinations from a (possibly huge) parameter space instead of trying all of them. Faster when the grid is too large to search exhaustively.

**Data leakage**: when information from outside the training fold sneaks into the process that produces your CV score. The score looks great. It lies.

## 1. The Problem: One Split, One Lucky (or Unlucky) Number

Same model, same dataset, same code. Only the `random_state` used to split the data changes, ten times in a row.

In [ ]:
X, y = make_classification(
    n_samples=300, n_features=15, n_informative=6, n_redundant=4,
    n_classes=2, flip_y=0.08, class_sep=0.8, random_state=42
)

split_results = []
for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y
    )
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    split_results.append({"random_state": seed, "test_accuracy": round(acc, 3)})

split_df = pd.DataFrame(split_results)
split_df

In [ ]:
print("Accuracy range across 10 splits:", split_df["test_accuracy"].min(), "to", split_df["test_accuracy"].max())
print("Standard deviation:", round(split_df["test_accuracy"].std(), 4))
print()
print("Same model. Same data. Same code. An 11-point swing depending purely on")
print("which rows happened to land in the test set.")

If you'd only run this once, with `random_state=1`, you would have reported 77.8% and possibly rejected a model that was actually fine. Run it with `random_state=5` instead and you'd have reported 88.9% and shipped something you got lucky on. Neither number is wrong. Neither number is reliable on its own either.

## 2. The Fix: K-Fold Cross-Validation

Instead of one split, use five. Every row gets used for training in four of the folds and for validation in exactly one. `StratifiedKFold` keeps the class balance consistent across folds, important whenever you have imbalanced classes (see Day 19).

In [ ]:
clf = RandomForestClassifier(n_estimators=50, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y, cv=skf, scoring="accuracy")

print("CV scores across 5 folds:", np.round(cv_scores, 3))
print(f"Mean: {cv_scores.mean():.3f}")
print(f"Std:  {cv_scores.std():.3f}")
print()
print(f"Compare: 10 single splits ranged from {split_df['test_accuracy'].min()} to {split_df['test_accuracy'].max()}")
print(f"5-fold CV gives one number with a confidence band: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(range(10), split_df["test_accuracy"], color="#E53935", s=60, label="10 single train/test splits", zorder=3)
ax.axhline(split_df["test_accuracy"].mean(), color="#E53935", linestyle="--", alpha=0.5)

ax.scatter(range(10, 15), cv_scores, color="#00C853", s=60, label="5-fold CV scores", zorder=3)
ax.axhline(cv_scores.mean(), color="#00C853", linestyle="--", alpha=0.5)

ax.set_xlabel("Run")
ax.set_ylabel("Accuracy")
ax.set_title("Single Splits (noisy) vs Cross-Validation (stable)")
ax.legend()
plt.tight_layout()
plt.show()

The red dots scatter all over the place. The green dots cluster tighter around their own mean. CV doesn't eliminate randomness, it averages it out enough that the final number actually means something when you compare two models.

## 3. Tuning Properly: GridSearchCV

Default hyperparameters are a starting guess, not a guarantee. `GridSearchCV` tries every combination you give it, scoring each one with cross-validation, and hands you the best.

Switching to `SVC` here on purpose: it's a model where the default hyperparameters (`C=1`, `gamma='scale'`) genuinely leave performance on the table, unlike Random Forest, which tends to do reasonably well out of the box.

In [ ]:
X2, y2 = make_classification(
    n_samples=400, n_features=20, n_informative=8, n_redundant=6,
    n_classes=2, flip_y=0.05, class_sep=0.9, random_state=42
)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=42, stratify=y2
)

default_clf = SVC(random_state=42)
default_clf.fit(X2_train, y2_train)
default_acc = accuracy_score(y2_test, default_clf.predict(X2_test))

skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
default_cv = cross_val_score(default_clf, X2_train, y2_train, cv=skf2, scoring="accuracy")

print("Default SVC (C=1, gamma='scale')")
print("CV score:        ", round(default_cv.mean(), 3))
print("Test accuracy:    ", round(default_acc, 3))

In [ ]:
param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.001, 0.01, 0.1, 1],
    "kernel": ["rbf"],
}

start = time.time()
grid = GridSearchCV(SVC(random_state=42), param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X2_train, y2_train)
grid_time = time.time() - start

tuned_acc = accuracy_score(y2_test, grid.best_estimator_.predict(X2_test))

print(f"GridSearchCV fit time: {grid_time:.2f}s")
print(f"Combinations tried: {len(grid.cv_results_['params'])}")
print(f"Best params: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_:.3f}")
print(f"Tuned test accuracy: {tuned_acc:.3f}")
print()
print(f"CV score moved from {default_cv.mean():.3f} to {grid.best_score_:.3f} just by searching the hyperparameter space.")

## 4. When the Grid Is Too Big: RandomizedSearchCV

`GridSearchCV` above tried 20 fixed combinations. Real hyperparameter spaces are often continuous and far larger than anything you can exhaustively check. `RandomizedSearchCV` samples a fixed budget of combinations from a distribution instead, so you can search a much wider space for roughly the same cost.

In [ ]:
param_dist = {
    "C": loguniform(0.01, 1000),
    "gamma": loguniform(0.0001, 10),
    "kernel": ["rbf"],
}

start = time.time()
random_search = RandomizedSearchCV(
    SVC(random_state=42), param_distributions=param_dist,
    n_iter=20, cv=5, scoring="accuracy", random_state=42, n_jobs=-1
)
random_search.fit(X2_train, y2_train)
random_time = time.time() - start

random_tuned_acc = accuracy_score(y2_test, random_search.best_estimator_.predict(X2_test))

print(f"RandomizedSearchCV fit time: {random_time:.2f}s, iterations: 20")
print(f"Best params: C={random_search.best_params_['C']:.3f}, gamma={random_search.best_params_['gamma']:.4f}")
print(f"Best CV score: {random_search.best_score_:.3f}")
print(f"Test accuracy: {random_tuned_acc:.3f}")
print()
print(f"GridSearchCV (20 fixed combos, small grid):       best CV = {grid.best_score_:.3f}")
print(f"RandomizedSearchCV (20 sampled combos, huge space): best CV = {random_search.best_score_:.3f}")

With the same budget of 20 fits, GridSearchCV exhaustively covers a small grid you defined by hand. RandomizedSearchCV samples 20 points from a continuous space thousands of times larger. Neither one is strictly better. GridSearchCV is the right call when you already have a good idea of the useful range. RandomizedSearchCV earns its place when the space is too big to grid out, or when you're exploring blind.

## 5. The Trap: Data Leakage During Tuning

Here's a mistake that produces a great-looking CV score that falls apart in production: selecting features using the *entire* dataset, including rows that later land in your validation folds, before running cross-validation.

To make the effect impossible to miss, this dataset has 1,000 features, only 3 of which are actually informative, and just 100 rows.

In [ ]:
X3, y3 = make_classification(
    n_samples=100, n_features=1000, n_informative=3, n_redundant=0, n_repeated=0,
    n_classes=2, flip_y=0.1, class_sep=0.8, random_state=42
)

skf3 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# WRONG: select the best features using ALL the data, before cross-validation even starts
selector_leaky = SelectKBest(f_classif, k=10)
X_selected_leaky = selector_leaky.fit_transform(X3, y3)  # has already seen every label, including future "validation" rows
leaky_scores = cross_val_score(SVC(), X_selected_leaky, y3, cv=skf3, scoring="accuracy")

print("Leaky approach: select features on the full dataset, THEN cross-validate")
print("CV scores:", np.round(leaky_scores, 3))
print("Mean:", round(leaky_scores.mean(), 3))

In [ ]:
# CORRECT: feature selection lives inside a Pipeline, refit on only the training fold each time
correct_pipeline = Pipeline([
    ("select", SelectKBest(f_classif, k=10)),
    ("svc", SVC()),
])
correct_scores = cross_val_score(correct_pipeline, X3, y3, cv=skf3, scoring="accuracy")

print("Correct approach: feature selection inside the Pipeline, refit per fold")
print("CV scores:", np.round(correct_scores, 3))
print("Mean:", round(correct_scores.mean(), 3))

print(f"\nThe leaky version reports a CV score {leaky_scores.mean() - correct_scores.mean():.3f} higher than reality.")
print("That gap is the model getting credit for already having seen the answers.")

The leaky score looks like a working model. The honest score reveals there's barely any real signal in this particular dataset, which is the truth. Anything that learns from your data, scalers, feature selectors, imputers, target encoders, has to live inside a `Pipeline` and refit on each training fold separately. The moment a preprocessing step sees the full dataset before cross-validation starts, your validation score stops being a validation score.

## 6. Practice: Spot the Leakage

Four setups below. Before running the next cell, decide for each one: does it leak information, or is it clean?

In [ ]:
scenarios = [
    {"name": "Setup A", "description": "Scale features on the full dataset, then run 5-fold CV on the scaled data."},
    {"name": "Setup B", "description": "Build a Pipeline with StandardScaler and LogisticRegression, then run cross_val_score on the raw data."},
    {"name": "Setup C", "description": "Use SelectKBest to pick the top 20 features using the full dataset, then GridSearchCV on those features."},
    {"name": "Setup D", "description": "Split into train/test first. Fit a Pipeline (imputer + scaler + model) using cross_val_score on the train set only for tuning."},
]

for s in scenarios:
    print(f"{s['name']}: {s['description']}")

**Your turn.** Write LEAKAGE or CLEAN for each setup before checking the solution below.

- Setup A: 
- Setup B: 
- Setup C: 
- Setup D: 

### Solution

In [ ]:
answers = [
    {"name": "Setup A", "verdict": "LEAKAGE", "why": "The scaler's mean and std were computed using rows that later land in validation folds."},
    {"name": "Setup B", "verdict": "CLEAN", "why": "The pipeline refits the scaler on only the training fold each time."},
    {"name": "Setup C", "verdict": "LEAKAGE", "why": "Feature selection saw the labels of rows that end up in validation folds during the grid search."},
    {"name": "Setup D", "verdict": "CLEAN", "why": "The test set was never touched until the very end, and train-only CV is leakage-free."},
]

for a in answers:
    print(f"{a['name']}: {a['verdict']} -- {a['why']}")

Setups A and C share the same root mistake: a transformation that learns from data (scaling stats, feature rankings) was fit before the train/validation split existed for that fold. Setups B and D fix it the same way too: wrap everything that learns from data inside a `Pipeline`, and let cross-validation refit that pipeline fresh on each fold.

## 7. Try It Yourself

No solution provided here. Make these changes and see what happens:

1. In Section 2, change `n_splits=5` to `n_splits=10`. Does the standard deviation of the CV scores go up or down? Why might that be?
2. In Section 3, add `"kernel": ["linear", "rbf"]` to `param_grid` and rerun. How many combinations does GridSearchCV try now, and does the best kernel change?
3. In Section 5, change `n_informative` from 3 to 30 (keeping 1,000 total features). Does the leakage gap get bigger or smaller? What does that tell you about when leakage matters most?

## Self-Check Before Day 21

You're ready to move on if you can answer these without scrolling back up:

1. Why can two runs of the exact same model on the exact same data report different accuracy?
2. What does StratifiedKFold do differently from a plain KFold split?
3. When would you reach for RandomizedSearchCV instead of GridSearchCV?
4. Name one preprocessing step that causes leakage if it's fit before cross-validation instead of inside a Pipeline.
5. A model reports a 0.95 CV score and a 0.70 test score on a held-out set it never touched during tuning. What's your first suspicion?

If any of these feel shaky, re-run the relevant section above before starting Day 21.

## What's Next

Tomorrow, Day 21: Gradient Boosting with XGBoost. We'll use the GridSearchCV and cross-validation skills from today to tune a model that wins a large share of structured-data competitions, and look at why boosting trees sequentially behaves so differently from the Random Forest's bagging approach.

Full series repo: github.com/VaishnaviJagtap18/42-days-aiml-challenge

#42DaysOfML #MachineLearning #MLEngineer #Python #DataScience